# Exercise 2. Tune for Low-Resource Languages with LoRa
NLP for social good is not just about reducing harmful outputs; it is also about making AI accessible across languages, not only English. Low- and medium-resource languages, from Nigerian Pidgin to Danish, are often left behind. 

```{figure} ../figures/class8/neural-space-low-resource.png
---
name: neural-space-low-resource
width: 100%
---
Fig. borrowed from [NeuralSpace blogpost](https://medium.com/neuralspace/challenges-in-using-nlp-for-low-resource-languages-and-how-neuralspace-solves-them-54a01356a71b) by Felix Laumann
```

Fine-tuning LLMs can help, but it is costly. LoRA (Low-Rank Adaptation) offers a parameter-efficient alternative, reducing trainable parameters by up to 10,000 times. In other words, rather than training all 8 billion parameters of a model like [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B-Base), LoRA updates only a small fraction.

## 2.1 Intro to LoRa?
If you're interested in the math behind this (but in an intuitive way), I encourage you to read Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/i/138081202/a-brief-introduction-to-lora). You can also read the original paper by {cite:t}`hu_lora_2021`. 

## 2.2 Setup
For the code implementation, we'll use the [PEFT](https://huggingface.co/docs/peft/en/index) and [TRL](https://huggingface.co/docs/trl/en/index) library by Hugging Face
```bash
source .venv/bin/activate
pip install peft trl
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers datasets
```

Let's import:

In [57]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel

## 2.3 Load Model & Data
For today's exercise, we'll try to make a smaller version of `SmolLM2` good at English to Danish machine translation

:::{admonition} You can use LoRa for much more than Translation :)
:class: dropdown, tip
As a simple introduction to LoRA, we're doing machine translation, but you can use this approach for anything you'd like really - feel free to switch out the dataset for something you'd like. Or use this notebook as a inspiration for the exam :).

See also this tutorial for instruction-tuning a danish language model using QLoRA -> [Tutorial: Finetuning Language Models](https://www.foundationmodels.dk/blog/2024/02/02/tutorial-finetuning-language-models.html)
:::

We'll load a smaller version of `SmolLM2`:

In [58]:
model_id = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

We'll load the Danish-English translation set:

In [59]:
train_ds = load_dataset("Helsinki-NLP/opus-100", "da-en", split="train")

And downsample:

In [60]:
train_ds = train_ds.train_test_split(seed=24, test_size=2000)["test"]

Let's look at the only column, "translation" to see how it is structured: 

In [61]:
train_ds["translation"]

Column([{'da': 'Men den plan var ikke idiotsikker.', 'en': "Well, that plan wasn't foolproof."}, {'da': 'Det mener jeg, at Europa-Parlamentet bør notere sig, og det bør støtte ønsket om også på denne måde at deltage fuldt ud i det internationale samfund, for hvem ved, om domstolen i fremtiden netop kommer til at dømme de forbrydelser, der er blevet begået i landet.', 'en': 'Well, I feel that Parliament should take note and encourage this desire to become a fully paid-up member of the international community, also in this respect, because who knows, in the future the Court could operate precisely to try crimes committed in that country.'}, {'da': 'virkning på Deres blodsukkerkontrol. i', 'en': 'effect on your blood glucose control. ed'}, {'da': 'Du er så smuk.', 'en': "You're so beautiful."}, {'da': 'Jeg kom netop med to eksempler - en i Messinastrædet og en på Seinen for et par dage siden.', 'en': 'I gave just two examples: one on the Strait of Messina and the other on the River Seine 

Let's print a few:

In [62]:
for translation in train_ds["translation"][:5]:
    print(f"EN: {translation['en']}")
    print(f"DA: {translation['da']}")
    print()

EN: Well, that plan wasn't foolproof.
DA: Men den plan var ikke idiotsikker.

EN: Well, I feel that Parliament should take note and encourage this desire to become a fully paid-up member of the international community, also in this respect, because who knows, in the future the Court could operate precisely to try crimes committed in that country.
DA: Det mener jeg, at Europa-Parlamentet bør notere sig, og det bør støtte ønsket om også på denne måde at deltage fuldt ud i det internationale samfund, for hvem ved, om domstolen i fremtiden netop kommer til at dømme de forbrydelser, der er blevet begået i landet.

EN: effect on your blood glucose control. ed
DA: virkning på Deres blodsukkerkontrol. i

EN: You're so beautiful.
DA: Du er så smuk.

EN: I gave just two examples: one on the Strait of Messina and the other on the River Seine a few days ago.
DA: Jeg kom netop med to eksempler - en i Messinastrædet og en på Seinen for et par dage siden.



## 1.3 Prompt Templating
We want a `prompt` column that inserts the English sentence as the `Source` and the Danish translation `Target` in the format by {cite:t}`alves_steering_2023`:
```{figure} ../figures/class8/prompt-template-alves.png
---
name: prompt-template-alves
width: 80%
---
Prompt template by {cite:t}`alves_steering_2023`
```
X should be "English" and Y should be "Danish" in our context.

### Your Turn: Formatting the Prompt
:::{admonition} HANDS-ON
:class: red
1. Create a function called `def format_prompt(example)`
    - It should process a single row `example` in our dataset
    - Format a prompt as the template above using the `translation` column
    - Return a dictionary entry `{"prompt": prompt}`

2. Test the function on a single example in `train_ds`, printing the prompt!
:::

#### Solution

In [69]:
# define function
def format_prompt(example):
    translation = example["translation"]
    prompt = f"Translate the source text from English to Danish. Source: {translation['en']} Target: {translation['da']}"

    prompt_row = {"prompt": prompt}
    return prompt_row

# test on one example
example = train_ds[0]
print(format_prompt(example)["prompt"])

Translate the source text from English to Danish. Source: Well, that plan wasn't foolproof. Target: Men den plan var ikke idiotsikker.


### Adding a Prompt Column 
We can now add the prompt column to our ds using our new `format_prompt` column:

In [70]:
train_ds = train_ds.map(format_prompt, batched=False)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [71]:
train_ds["prompt"][0]

"Translate the source text from English to Danish. Source: Well, that plan wasn't foolproof. Target: Men den plan var ikke idiotsikker."

## 1.4 LoRa Configuration